# Congressional Trading Patterns -- A Data Analysis

**Dataset:** U.S. House of Representatives STOCK Act Disclosures  
**Source:** House Stock Watcher (housestockwatcher.com) -- real public filings  
**Coverage:** 2012 to present  

This notebook applies data manipulation (NumPy / Pandas), exploratory data analysis,
linear regression, logistic regression, and time series analysis to real congressional
equity trading disclosures filed under the STOCK Act.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarkArenSangha/Polymarket-Project/blob/main/20260508_Data_Modeling_Final_Project_Mark_Aren_Sangha.ipynb)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests'], check=False)

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_squared_error, r2_score,
                             accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
print('Ready.')


## 1. Dataset Documentation

**Source:** House Stock Watcher aggregates mandatory STOCK Act filings from the U.S. House Clerk's office.  
**URL:** `https://house-stock-watcher-data.s3-us-west-2.amazonaws.com/data/all_transactions.json`  
**Access:** Public JSON, no authentication required.  

**Variables:**

| Column | Description |
|--------|-------------|
| `transaction_date` | Date the trade was executed |
| `disclosure_date` | Date the member filed the disclosure |
| `representative` | House member name |
| `party` | Political party |
| `ticker` | Stock ticker |
| `type` | purchase / sale / sale_partial |
| `amount` | Reported dollar range (e.g. `$15,001 - $50,000`) |

**Why this dataset?**  
The STOCK Act (2012) requires House members to publicly disclose equity trades within 45 days.
This dataset is real, publicly verifiable, contains thousands of rows with multiple features,
and supports all required analysis types: time series (trades over time), linear regression
(predicting disclosure delay), and logistic regression (predicting buy vs. sell).


In [ ]:
URL = 'https://house-stock-watcher-data.s3-us-west-2.amazonaws.com/data/all_transactions.json'
print(f'Downloading from {URL} ...')
resp = requests.get(URL, timeout=60)
resp.raise_for_status()
df_raw = pd.DataFrame(resp.json())
print(f'Raw dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print('Columns:', list(df_raw.columns))
df_raw.head(3)


## 2. Data Cleaning and Preparation


In [ ]:
df = df_raw.copy()

# Parse dates
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')
df['disclosure_date']  = pd.to_datetime(df['disclosure_date'],  errors='coerce')

# Drop missing critical fields
n0 = len(df)
df = df.dropna(subset=['transaction_date', 'disclosure_date', 'type', 'amount', 'representative'])
print(f'Dropped {n0 - len(df):,} rows with missing critical fields.')

# Standardise trade type
df['type_clean'] = df['type'].str.lower().str.strip()
df = df[df['type_clean'].isin(['purchase', 'sale', 'sale_partial', 'sale (partial)'])]
df['trade_type'] = df['type_clean'].replace({'sale_partial': 'sale', 'sale (partial)': 'sale'})

# Parse amount to numeric midpoint
def parse_amount(s):
    s = str(s).replace('$', '').replace(',', '').replace('+', '').strip()
    if 'over' in s.lower():
        parts = [t for t in s.split() if t.replace('.', '').isdigit()]
        return float(parts[0]) if parts else np.nan
    nums = []
    for tok in s.replace('-', ' ').split():
        try:
            nums.append(float(tok))
        except ValueError:
            pass
    if len(nums) >= 2:
        return (nums[0] + nums[1]) / 2
    elif len(nums) == 1:
        return nums[0]
    return np.nan

df['amount_mid'] = df['amount'].apply(parse_amount)

# Disclosure delay
df['disclosure_delay'] = (df['disclosure_date'] - df['transaction_date']).dt.days
n1 = len(df)
df = df[(df['disclosure_delay'] >= 0) & (df['disclosure_delay'] <= 730)]
print(f'Removed {n1 - len(df):,} rows with implausible disclosure delays.')

# Cap amount at 99th percentile
cap = df['amount_mid'].quantile(0.99)
n2 = len(df)
df = df[df['amount_mid'].notna() & (df['amount_mid'] <= cap)]
print(f'Removed {n2 - len(df):,} extreme amount outliers (cap = ${cap:,.0f}).')

# Party
if 'party' in df.columns:
    df['party'] = df['party'].astype(str).str.strip().str.title()
    df['party'] = df['party'].replace({'D': 'Democrat', 'R': 'Republican',
                                       'Democrat': 'Democrat', 'Republican': 'Republican'})
    df = df[df['party'].isin(['Democrat', 'Republican'])]
else:
    df['party'] = 'Unknown'

# Feature engineering
df['year']        = df['transaction_date'].dt.year
df['month']       = df['transaction_date'].dt.month
df['is_purchase'] = (df['trade_type'] == 'purchase').astype(int)
df['party_enc']   = (df['party'] == 'Republican').astype(int)
df['log_amount']  = np.log1p(df['amount_mid'])

print(f'\nFinal dataset: {len(df):,} rows x {df.shape[1]} columns')
print(f'Date range: {df["transaction_date"].min().date()} -- {df["transaction_date"].max().date()}')
print(f'Unique representatives: {df["representative"].nunique()}')
print(f'\nRemaining missing values in key columns:')
print(df[['disclosure_delay', 'amount_mid', 'party', 'year']].isnull().sum())


## 3. Summary Statistics

Descriptive statistics computed with Pandas and NumPy across the key numeric variables.


In [ ]:
print('=== .describe() -- Pandas ===')
print(df[['disclosure_delay', 'amount_mid', 'year', 'month']].describe().round(2).to_string())

delays  = df['disclosure_delay'].values
amounts = df['amount_mid'].values

print('\n=== NumPy statistics ===')
print(f'Total trades                 : {len(df):,}')
print(f'Unique representatives       : {df["representative"].nunique()}')
print(f'Purchases                    : {df["is_purchase"].sum():,} ({df["is_purchase"].mean()*100:.1f}%)')
print(f'Sales                        : {(df["trade_type"]=="sale").sum():,}')
if df['party'].nunique() > 1:
    print(f'Democrat trades              : {(df["party"]=="Democrat").sum():,}')
    print(f'Republican trades            : {(df["party"]=="Republican").sum():,}')
print(f'\nDisclosure delay (days)')
print(f'  Median : {np.median(delays):.0f}')
print(f'  Mean   : {np.mean(delays):.1f}')
print(f'  Std    : {np.std(delays):.1f}')
print(f'  Late (>45 days): {(delays > 45).mean()*100:.1f}%')
print(f'\nTrade amount midpoint (USD)')
print(f'  Median : ${np.median(amounts):,.0f}')
print(f'  Mean   : ${np.mean(amounts):,.0f}')
print(f'  Std    : ${np.std(amounts):,.0f}')


## 4. Exploratory Data Analysis

### Graph 1 -- Trade Count by Party and Type (Bar Chart)

This grouped bar chart shows how many purchases and sales were reported by Democrat
and Republican members. It reveals whether one party trades more actively or skews
toward buying versus selling.


In [ ]:
pt = (df.groupby(['party', 'trade_type'])
       .size()
       .unstack(fill_value=0)
       .reindex(columns=['purchase', 'sale'], fill_value=0))

x, w = np.arange(len(pt)), 0.35
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, pt['purchase'], w, label='Purchase', color='steelblue')
b2 = ax.bar(x + w/2, pt['sale'],     w, label='Sale',     color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(pt.index, fontsize=12)
ax.set_ylabel('Number of Trades')
ax.set_title('Trade Count by Party and Type (STOCK Act Disclosures)')
ax.legend()
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 30, f'{h:,.0f}',
            ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()


### Graph 2 -- Monthly Trade Volume Over Time (Line Chart)

This line chart plots the number of trades disclosed each month alongside a 12-month
rolling average. It shows whether congressional trading has grown since the STOCK Act
and identifies any event-driven spikes.


In [ ]:
monthly = (df.groupby(df['transaction_date'].dt.to_period('M'))
             .size()
             .reset_index(name='count'))
monthly['date'] = monthly['transaction_date'].dt.to_timestamp()
monthly = monthly.sort_values('date')
roll12 = monthly.set_index('date')['count'].rolling(12, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(monthly['date'], monthly['count'], color='steelblue', lw=1.2,
        alpha=0.6, label='Monthly trades')
ax.plot(roll12.index, roll12.values, color='tomato', lw=2.5,
        linestyle='--', label='12-month rolling mean')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45)
ax.set_ylabel('Trades per Month')
ax.set_title('Monthly Congressional Trade Disclosures Over Time')
ax.legend()
plt.tight_layout()
plt.show()


### Graph 3 -- Disclosure Delay Distribution (Histogram)

The STOCK Act mandates disclosure within 45 days. This histogram shows how members
cluster their filings relative to that deadline and what fraction files late.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['disclosure_delay'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(45, color='tomato', lw=2.5, linestyle='--', label='45-day STOCK Act deadline')
med = np.median(delays)
ax.axvline(med, color='green', lw=2, linestyle=':', label=f'Median = {med:.0f} days')
ax.set_xlabel('Days Between Trade and Disclosure')
ax.set_ylabel('Number of Trades')
ax.set_title('Distribution of STOCK Act Disclosure Delays')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Late filings (> 45 days): {(delays > 45).mean()*100:.1f}% of all trades.')


### Graph 4 -- Trade Amount by Party (Box Plot)

Box plots compare the distribution of reported trade amounts between Democrat and
Republican members. The midpoint of each disclosed dollar range is used as the estimate.


In [ ]:
dem_amt = df[df['party'] == 'Democrat']['amount_mid'].values
rep_amt = df[df['party'] == 'Republican']['amount_mid'].values

fig, ax = plt.subplots(figsize=(7, 5))
bp = ax.boxplot([dem_amt, rep_amt],
                labels=['Democrat', 'Republican'],
                patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('tomato')
ax.set_ylabel('Trade Amount Midpoint (USD)')
ax.set_title('Trade Amount Distribution by Party')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()
print(f'Democrat   -- Median: ${np.median(dem_amt):,.0f}  Mean: ${np.mean(dem_amt):,.0f}')
print(f'Republican -- Median: ${np.median(rep_amt):,.0f}  Mean: ${np.mean(rep_amt):,.0f}')


### Graph 5 -- Trade Amount vs. Disclosure Delay (Scatter Plot)

This scatter plot explores whether larger trades are disclosed faster or slower,
with a linear trend line fitted using SciPy. The correlation coefficient indicates
the strength and direction of the relationship.


In [ ]:
samp = df[['amount_mid', 'disclosure_delay']].dropna()
if len(samp) > 4000:
    samp = samp.sample(4000, random_state=42)

slope, intercept, r_val, p_val, _ = stats.linregress(
    samp['amount_mid'], samp['disclosure_delay'])
x_line = np.linspace(samp['amount_mid'].min(), samp['amount_mid'].max(), 200)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(samp['amount_mid'], samp['disclosure_delay'],
           alpha=0.2, s=10, color='steelblue')
ax.plot(x_line, slope * x_line + intercept, color='tomato', lw=2,
        label=f'Trend  r = {r_val:.3f}  p = {p_val:.3f}')
ax.axhline(45, color='green', lw=1.5, linestyle='--', label='45-day deadline')
ax.set_xlabel('Trade Amount Midpoint (USD)')
ax.set_ylabel('Disclosure Delay (days)')
ax.set_title('Trade Amount vs. Disclosure Delay')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()


## 5. Time Series Analysis

We fit a linear trend to annual trade counts to quantify the growth in congressional
trading disclosures since the STOCK Act was enacted in 2012, and plot year-over-year change.


In [ ]:
annual = (df[df['year'].between(2013, 2024)]
           .groupby('year').size()
           .reset_index(name='trades'))

X_yr = annual['year'].values.reshape(-1, 1)
y_yr = annual['trades'].values
trend_m = LinearRegression().fit(X_yr, y_yr)
y_trend = trend_m.predict(X_yr)

annual['yoy'] = annual['trades'].diff()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(annual['year'], annual['trades'], color='steelblue', edgecolor='white', label='Annual trades')
axes[0].plot(annual['year'], y_trend, color='tomato', lw=2.5, linestyle='--', label='Linear trend')
axes[0].set_title('Annual Congressional Trades (with trend)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Total Trades')
axes[0].legend()
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(2))
plt.setp(axes[0].get_xticklabels(), rotation=45)

colors_yoy = ['tomato' if v < 0 else 'steelblue' for v in annual['yoy'].fillna(0)]
axes[1].bar(annual['year'], annual['yoy'].fillna(0), color=colors_yoy, edgecolor='white')
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title('Year-Over-Year Change in Trade Count')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Change')
axes[1].xaxis.set_major_locator(mticker.MultipleLocator(2))
plt.setp(axes[1].get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()

r2_ts = r2_score(y_yr, y_trend)
print(f'Annual trend: +{trend_m.coef_[0]:.0f} additional trades per year')
print(f'R-squared of linear trend: {r2_ts:.3f}')


## 6. Linear Regression -- Predicting Disclosure Delay

**Target:** `disclosure_delay` (days between trade and filing)  
**Features:** year, month, log-transformed trade amount, party (encoded), trade type  

We test whether these observable features can predict how quickly a member discloses.


In [ ]:
feats_lr = ['year', 'month', 'log_amount', 'party_enc', 'is_purchase']
df_lr = df[feats_lr + ['disclosure_delay']].dropna()
X_lr = df_lr[feats_lr].values
y_lr = df_lr['disclosure_delay'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_lr, y_lr, test_size=0.2, random_state=42)
lr = LinearRegression().fit(X_tr, y_tr)
y_pred_lr = lr.predict(X_te)

rmse = np.sqrt(mean_squared_error(y_te, y_pred_lr))
r2   = r2_score(y_te, y_pred_lr)
print('=== Linear Regression: Predicting Disclosure Delay ===')
print(f'R-squared (test) : {r2:.4f}')
print(f'RMSE (days)      : {rmse:.2f}')
print('\nCoefficients:')
for f, c in zip(feats_lr, lr.coef_):
    print(f'  {f:<15}  {c:+.4f}')
print(f'  {"intercept":<15}  {lr.intercept_:+.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_pred_lr, y_te - y_pred_lr, alpha=0.15, s=8, color='steelblue')
axes[0].axhline(0, color='tomato', lw=2)
axes[0].set_xlabel('Predicted Delay (days)')
axes[0].set_ylabel('Residual (days)')
axes[0].set_title('Residual Plot')
mn, mx = y_te.min(), y_te.max()
axes[1].scatter(y_te, y_pred_lr, alpha=0.15, s=8, color='steelblue')
axes[1].plot([mn, mx], [mn, mx], color='tomato', lw=2, label='Perfect fit')
axes[1].set_xlabel('Actual Delay (days)')
axes[1].set_ylabel('Predicted Delay (days)')
axes[1].set_title(f'Actual vs. Predicted  (R-sq = {r2:.3f})')
axes[1].legend()
plt.tight_layout()
plt.show()
print('\nInterpretation: A low R-squared means disclosure timing is largely idiosyncratic --')
print('not well explained by party, trade size, or calendar features alone.')


## 7. Logistic Regression -- Predicting Trade Direction (Buy vs. Sell)

**Target:** `is_purchase` (1 = purchase, 0 = sale)  
**Features:** year, month, log trade amount, party, disclosure delay  

We test whether party affiliation, trade size, and timing predict whether a member
is buying or selling.


In [ ]:
feats_log = ['year', 'month', 'log_amount', 'party_enc', 'disclosure_delay']
df_log = df[feats_log + ['is_purchase']].dropna()
X_log = df_log[feats_log].values
y_log = df_log['is_purchase'].values

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_log, y_log, test_size=0.2, random_state=42, stratify=y_log)
scaler = StandardScaler()
X_tr2s = scaler.fit_transform(X_tr2)
X_te2s  = scaler.transform(X_te2)

log_m = LogisticRegression(max_iter=500, random_state=42)
log_m.fit(X_tr2s, y_tr2)
y_pred_log = log_m.predict(X_te2s)

acc = accuracy_score(y_te2, y_pred_log)
baseline = max(y_log.mean(), 1 - y_log.mean())
print('=== Logistic Regression: Buy vs. Sell ===')
print(f'Accuracy (test) : {acc:.4f}  ({acc*100:.1f}%)')
print(f'Baseline        : {baseline*100:.1f}% (majority class)')
print('\nClassification Report:')
print(classification_report(y_te2, y_pred_log, target_names=['Sale', 'Purchase']))
print('Coefficients (standardised):')
for f, c in zip(feats_log, log_m.coef_[0]):
    print(f'  {f:<20}  {c:+.4f}')

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_te2, y_pred_log),
    display_labels=['Sale', 'Purchase']
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix -- Logistic Regression')
plt.tight_layout()
plt.show()
print('\nInterpretation: Accuracy near the baseline indicates that buy/sell decisions')
print('are not strongly predicted by these observable features.')


## 8. Conclusions and Limitations

### Summary of Findings

| Analysis | Finding |
|----------|---------|
| Dataset | Real STOCK Act filings, U.S. House members, 2012 -- present |
| Late disclosures (> 45 days) | See Section 3 output |
| Annual growth trend | See Section 5 output |
| Linear regression | Low R-squared -- disclosure timing is idiosyncratic |
| Logistic regression | Near majority-class baseline |

### Interpretation
- Congressional trade volume has grown substantially since STOCK Act passage in 2012.
- A meaningful fraction of members file disclosures after the 45-day legal deadline.
- Neither disclosure timing nor trade direction is well-predicted by party, trade size,
  or year, suggesting individual-level factors dominate both decisions.

### Limitations
- **Amount is a range, not exact** -- midpoint approximation introduces measurement error.
- **House members only** -- Senate requires a separate dataset.
- **No stock return data** -- profitability cannot be assessed.
- **Party field is missing for some members** -- those rows were excluded.

### Data Source
All data is sourced from mandatory public STOCK Act filings aggregated by
House Stock Watcher (housestockwatcher.com). No data was fabricated.
